<h4 align='center'>AEROPULSE</h4>
<h5 align='center'>AIRLINE OPERATIONS, DELAY & RELIABILITY ANALYTICS</h5>

<h5 align='center'>NOTEBOOK 05 - FEATURE ENGINEERING</h5>

**Objective:** create business-oriented analytical features from the validated and transformed dataset..


**Importance:** Features must have a clear analytical purpose. Do not create derived columns merely to increase column count.

### Load Transformed Data

In [2]:
from pathlib import Path

import pandas as pd
import numpy as np

PROJECT_PATH = Path.cwd().parent

CLEANED_PATH = PROJECT_PATH / "04_Cleaned_Data"

TRANSFORMED_FILE = (
    CLEANED_PATH /
    "Aeropulse_Flights_January_2024_Cleaned.csv"
)

df = pd.read_csv(
    TRANSFORMED_FILE,
    low_memory=False
)

df["FlightDate"] = pd.to_datetime(
    df["FlightDate"],
    errors="coerce"
)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))

Rows: 547,271
Columns: 93


### Feature Engineering Inventory

In [3]:
feature_plan = pd.DataFrame({
    "Feature": [
        "Flight_Year",
        "Flight_Month",
        "Flight_Day",
        "Flight_Day_Name",
        "Route"
    ],

    "Business_Purpose": [
        "Year-level comparison",
        "Period-level analysis",
        "Daily operational analysis",
        "Day-od-week analysis",
        "Route-level reliability analysis"
    ]
})

display(feature_plan)

,Feature,Business_Purpose
0,Flight_Year,Year-level comparison
1,Flight_Month,Period-level analysis
2,Flight_Day,Daily operational analysis
3,Flight_Day_Name,Day-od-week analysis
4,Route,Route-level reliability analysis


### Cancellation Status

In [18]:
if "Cancelled" in df.columns:

    df["Cancellation_Status"] = np.where(
        df["Cancelled"] == 1,
        "Cancelled",
        "Completed"
    )

    print(
        df["Cancellation_Status"]
        .value_counts(dropna=False)
    )

Cancellation_Status
Completed    526882
Cancelled     20389
Name: count, dtype: int64


### Diversion Status

In [23]:
if "DepDel15" in df.columns :

    df["Departure_Delay_Status"] = np.select(
        [
            df["DepDel15"] == 1,
            df["DepDel15"] == 0
        ],
        [
            "15+ Minute Delay",
            "Less Than 15 Minutes Delay"
        ],
        default="Unknown"
    )

    print(
        df["Departure_Delay_Status"]
        .value_counts(dropna=False)
    )

Departure_Delay_Status
Less Than 15 Minutes Delay    405154
15+ Minute Delay              122259
Unknown                        19858
Name: count, dtype: int64


### Arrival Delay Status

In [21]:
if "ArrDel15" in df.columns :

    df["Arrival_Delay_Status"] = np.select(
        [
            df["ArrDel15"] == 1,
            df["ArrDel15"] == 0
        ],
        [
            "15+ Minute Delay",
            "Less Than 15 Minutes Delay"
        ],
        default="Unknown"
    )

    print(
        df["Arrival_Delay_Status"]
        .value_counts(dropna=False)
    )

Arrival_Delay_Status
Less Than 15 Minutes Delay    398960
15+ Minute Delay              126410
Unknown                        21901
Name: count, dtype: int64


### Delay Magnitude Classification

In [22]:
if "ArrDelayMinutes" in df.columns:

    df["Arrival_Delay_Band"] = pd.cut(
        df["ArrDelayMinutes"],
        bins=[
            -np.inf,
            0,
            15,
            30,
            60,
            120,
            np.inf
        ],
        labels=[
            "Early / On Time",
            "0-15 Minutes",
            "15-30 Minutes",
            "30-60 Minutes",
            "60-120 Minutes",
            "120+ Minutes"
        ]
    )

### Delay Cause Total

In [11]:
delay_cause_columns = [
    "CarrierDalay",
    "WeatherDelay",
    "NASDelay",
    "SecurityDelay",
    "LateAircraftDelay"
]

available_delay_causes = [
    column 
    for column in delay_cause_columns
    if column in df.columns
]

if available_delay_causes:
   print(
       "Available delay-cause columns:",
        available_delay_causes
    )

else:

    print(
        "No delay-cause columns available;"
        "feature not created."
    )

Available delay-cause columns: ['WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay']


### Feature Inventory

In [16]:
# List of derived features expected in dataset
derived_columns = [
    "Flight_Year",
    "Flight_Month",
    "Flight_Day",
    "Flight_Day_Name",
    "Route",
    "Cancellation_Status",
    "Diversion_Status",
    "Departure_Delay_Status",
    "Arrival_Delay_Status",
    "Departure_Delay_Band",
    "Total_Reported_Delay_Cause_Minutes"
]

# check which derived features actually exist in df
available_derived_columns = [
    column 
    for column in derived_columns
    if column in df.columns
]

# create feature inventory
feature_inventory = pd.DataFrame({
    "Derived_Feature": available_derived_columns,
    "Purpose": [
        "Analytical representation / business segmentation"
        for _ in available_derived_columns
    ]
})

display(feature_inventory)


,Derived_Feature,Purpose


###  Save Feature Dataset

In [4]:
FEATURED_FILE = (
    CLEANED_PATH /
    "AeroPulse_Flights_Feature_Engineered.csv"
)

df.to_csv(
    FEATURED_FILE,
    index=False
)

print("Feature-engineered dataset saved:")
print(FEATURED_FILE.resolve())

Feature-engineered dataset saved:
D:\DA_32 PYTHON\Aeropulse_Airline_Analytics\notebooks\04_Cleaned_Data\AeroPulse_Flights_Feature_Engineered.csv


### Feature Engineering Audit

In [17]:
feature_audit  = pd.DataFrame({
    "Feature": derived_columns,
    "Type": [
        "Derived analytical feature"
        for _ in derived_columns
    ],
    "Status": [
        "CREATED"
        for _ in derived_columns
    ]
})
audit_file = (
    PROJECT_PATH /
    "02_Documentation" /
    "05_Feature_Engineering_Audit.xlsx"
)

feature_audit.to_excel(
    audit_file,
    index=False
)

print("Feature audit saved:")
print(audit_file.resolve())


Feature audit saved:
D:\DA_32 PYTHON\Aeropulse_Airline_Analytics\notebooks\02_Documentation\05_Feature_Engineering_Audit.xlsx


In [ ]:
df["FlightDate"].dt.Month_name()